<a href="https://colab.research.google.com/github/aryanpatel99/GEN-AI-PRACTICE/blob/main/Transformers_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
! pip install transformers datasets evaluate accelerate

In [8]:
from datasets import Dataset
data = {
    "text": [
        "I am so happy today",
        "This is the best day",
        "I feel terrible and sad",
        "This is a disaster",
        "I am feeling great",       # Added 'great'
        "What a wonderful morning", # Added 'wonderful'
        "I hate this movie",        # Added 'hate'
        "This is awful",            # Added 'awful'
        "I love this place",        # Added 'love'
        "I am depressed"            # Added 'depressed'
    ],
    "label": [1, 1, 0, 0, 1, 1, 0, 0, 1, 0]
}
dataset = Dataset.from_dict(data)

In [3]:
# Preprocess
# The next step is to load a DistilBERT tokenizer to preprocess the text field:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=16 , padding = "max_length")

tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [5]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
args = TrainingArguments(
    output_dir="my_awesome_model",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset
)

trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=0.4272165616353353, metrics={'train_runtime': 10.0816, 'train_samples_per_second': 14.879, 'train_steps_per_second': 1.488, 'total_flos': 620940931200.0, 'train_loss': 0.4272165616353353, 'epoch': 15.0})

In [9]:
test_sentences = ["I am feeling great!", "This is the worst day ever."]


for text in test_sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits
    prediction = logits.argmax().item()
    label = "Happy" if prediction == 1 else "Sad"
    print(f"Input: '{text}' -> Prediction: {label}")

Input: 'I am feeling great!' -> Prediction: Happy
Input: 'This is the worst day ever.' -> Prediction: Happy
